# Lab 1 — Can We Trust the Data?
## Data Cleaning and Pipeline Integrity

**Scenario:** A station wants a Precision Recruiting Assistant, but its CRM export contains duplicate engagements, school-name variants, missing identifiers, inconsistent dates, and impossible funnel values.

Your job is to turn unreliable activity records into an auditable school summary.

**Learning goals**

- Profile a dataset before modeling.
- Resolve entities without silently merging the wrong records.
- validate funnel logic: `contacts ≥ appointments ≥ qualified ≥ contracts`.
- Produce a clean table with explicit data-quality flags.

*This is a first-pass scaffold. The final version will be expanded after Labs 2 and 3 stabilize.*

> **Use your coding assistant as a teammate.** Give it the current cell, the self-check output, and the goal. Ask it to explain the smallest useful change rather than rewriting the notebook.

Suggested prompt:

> I am working in a classroom Jupyter notebook. Explain what this self-check is testing, then suggest the smallest edit to the marked variables. Do not change the data or the test.

In [ ]:
from IPython.display import display, Markdown

def check(name, condition, hint=""):
    try:
        passed = bool(condition)
    except Exception as exc:
        passed = False
        hint = f"{hint} ({type(exc).__name__}: {exc})"
    icon = "✅" if passed else "❌"
    print(f"{icon} {name}")
    if not passed and hint:
        print(f"   Hint: {hint}")
    return passed

def mission_header(text):
    display(Markdown(f"> **Mission checkpoint:** {text}"))

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)

raw = pd.DataFrame([
    ["E001", "Jefferson HS",       "2026-01-12", 40, 14, 9, 5, 8],
    ["E002", "JEFFERSON HIGH",     "01/20/2026", 32, 11, 7, 4, 6],
    ["E002", "JEFFERSON HIGH",     "01/20/2026", 32, 11, 7, 4, 6],  # duplicate
    ["E003", "Jefferson High School", "2026/02/03", 25, 10, 12, 3, 7], # impossible
    ["E004", "Lincoln High",       "2026-01-15", 55, 19, 8, 3, 9],
    ["E005", "LINCOLN HS",         "not recorded", 31, 10, 6, 2, 5],
    ["E006", "Washington High",    "2025-12-11", 35, 12, 9, 6, 8],
    ["E007", "Washington H.S.",    "2026-02-18", 30, 9, 7, 5, 7],
    ["E008", "Roosevelt High",     "2026-01-22", 28, 8, 5, 3, 6],
    [None,   "Roosevelt High",      "2026-02-14", 20, 7, 4, 2, 5],
    ["E010", "North County Tech",  "2026-02-28", -4, 5, 3, 1, 4],     # impossible
    ["E011", None,                  "2026-03-01", 18, 6, 4, 2, 4],
], columns=[
    "engagement_id", "school_name", "event_date", "contacts",
    "appointments", "qualified", "contracts", "recruiter_hours"
])

raw

## 1. Profile before fixing

Pause and predict: how many rows are duplicated? Which columns have missing values? Which rows violate the funnel?

In [ ]:
profile = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "missing": raw.isna().sum(),
    "unique": raw.nunique(dropna=True),
})
display(profile)
print("Exact duplicate rows:", raw.duplicated().sum())

## 2. Resolve school identities

Edit only `NAME_MAP`. Use one canonical name for each school. Do not use fuzzy matching blindly: similar names are not always the same entity.

In [ ]:
NAME_MAP = {
    # TODO: add the known variants.
    # "Jefferson HS": "Jefferson High",
}

clean = raw.copy()
clean["school_name_clean"] = clean["school_name"].replace(NAME_MAP)
clean[["school_name", "school_name_clean"]].drop_duplicates()

In [ ]:
expected_schools = {
    "Jefferson High", "Lincoln High", "Washington High",
    "Roosevelt High", "North County Tech"
}
observed_schools = set(clean["school_name_clean"].dropna())
check(
    "Known school variants resolve to five canonical schools",
    observed_schools == expected_schools,
    "Map all Jefferson, Lincoln, and Washington variants. Leave missing names missing."
)

## 3. Parse, deduplicate, and validate

Fill the marked choices. Keep rejected records in an audit table; never make them disappear without explanation.

In [ ]:
REMOVE_DUPLICATE_IDS = False  # TODO: change after inspecting E002
INVALID_DATE_POLICY = "keep"  # TODO: choose "flag" for this lab

clean["event_date_clean"] = pd.to_datetime(
    clean["event_date"], errors="coerce", format="mixed"
)

if REMOVE_DUPLICATE_IDS:
    clean = clean.drop_duplicates(subset="engagement_id", keep="first")

clean["missing_key"] = clean["engagement_id"].isna() | clean["school_name_clean"].isna()
clean["invalid_date"] = clean["event_date_clean"].isna()
clean["negative_value"] = (clean[["contacts", "appointments", "qualified", "contracts", "recruiter_hours"]] < 0).any(axis=1)
clean["invalid_funnel"] = ~(
    (clean["contacts"] >= clean["appointments"])
    & (clean["appointments"] >= clean["qualified"])
    & (clean["qualified"] >= clean["contracts"])
)
clean["is_valid"] = ~clean[["missing_key", "invalid_date", "negative_value", "invalid_funnel"]].any(axis=1)

clean[["engagement_id", "school_name_clean", "is_valid", "missing_key", "invalid_date", "negative_value", "invalid_funnel"]]

In [ ]:
check("Duplicate engagement IDs are removed", clean["engagement_id"].dropna().is_unique,
      "Set REMOVE_DUPLICATE_IDS after verifying which record to keep.")
check("Dates are parsed and invalid dates are flagged", clean["invalid_date"].sum() == 1,
      "Use errors='coerce' and preserve an invalid-date flag.")
check("At least two impossible records are detected", (~clean["is_valid"]).sum() >= 2,
      "Check missing keys, negative values, dates, and funnel order.")

## 4. Produce the model-ready school summary

Aggregate only valid rows. Add a transparent quality measure based on all source records, not just the records that survived.

In [ ]:
# TODO: complete this section in the final Lab 1 build.
valid = clean.loc[clean["is_valid"]].copy()

school_summary = (
    valid.groupby("school_name_clean", as_index=False)
    .agg(
        recruiter_hours=("recruiter_hours", "sum"),
        contacts=("contacts", "sum"),
        appointments=("appointments", "sum"),
        qualified=("qualified", "sum"),
        contracts=("contracts", "sum"),
    )
    .rename(columns={"school_name_clean": "school_name"})
)

quality = clean.groupby("school_name_clean")["is_valid"].mean().rename("data_quality")
school_summary = school_summary.merge(quality, left_on="school_name", right_index=True, how="left")
school_summary

In [ ]:
check("Summary has one row per school", school_summary["school_name"].is_unique)
check("Summary includes downstream outcomes", {"qualified", "contracts"}.issubset(school_summary.columns))
check("Quality scores stay between 0 and 1", school_summary["data_quality"].between(0, 1).all())

## Handoff to Lab 2

The recommendation system should consume a table like `school_summary`, plus operational fields such as access and distance.

**Reflection:** Which errors should block a recommendation? Which should merely reduce confidence?